In [ ]:
# Modelo RNN mínimo a nivel de carácter (Vanilla RNN)
# Escrito originalmente por Andrej Karpathy y adaptado para el curso en la Maestría de Ciencia de Datos
# Comentado y adaptado pr Alfredo Diaz para Google Colab

import numpy as np

# Entrada de datos
# Asegúrate de subir un archivo 'input.txt' con el texto para entrenar
from google.colab import files
uploaded = files.upload()

data = open('input.txt', 'r').read()  # Carga el texto desde el archivo
chars = list(set(data))  # Caracteres únicos
data_size, vocab_size = len(data), len(chars)
print('El texto tiene %d caracteres, de los cuales %d son únicos.' % (data_size, vocab_size))

# Diccionarios para codificar caracteres como números
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }


In [ ]:
# Hiperparámetros
hidden_size = 100      # Tamaño de la capa oculta
seq_length = 25        # Longitud de secuencia para entrenamiento
learning_rate = 1e-1   # Tasa de aprendizaje

# Inicialización de pesos del modelo
Wxh = np.random.randn(hidden_size, vocab_size) * 0.01
Whh = np.random.randn(hidden_size, hidden_size) * 0.01
Why = np.random.randn(vocab_size, hidden_size) * 0.01
bh = np.zeros((hidden_size, 1))  # Sesgo de la capa oculta
by = np.zeros((vocab_size, 1))   # Sesgo de la salida


In [ ]:
def lossFun(inputs, targets, hprev):
  """
  Calcula la pérdida y los gradientes para una secuencia.
  """
  xs, hs, ys, ps = {}, {}, {}, {}
  hs[-1] = np.copy(hprev)
  loss = 0

  # Propagación hacia adelante
  for t in range(len(inputs)):
    xs[t] = np.zeros((vocab_size,1))  # One-hot
    xs[t][inputs[t]] = 1
    hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh)
    ys[t] = np.dot(Why, hs[t]) + by
    ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))
    loss += -np.log(ps[t][targets[t], 0])

  # Retropropagación
  dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
  dbh, dby = np.zeros_like(bh), np.zeros_like(by)
  dhnext = np.zeros_like(hs[0])
  for t in reversed(range(len(inputs))):
    dy = np.copy(ps[t])
    dy[targets[t]] -= 1
    dWhy += np.dot(dy, hs[t].T)
    dby += dy
    dh = np.dot(Why.T, dy) + dhnext
    dhraw = (1 - hs[t] * hs[t]) * dh
    dbh += dhraw
    dWxh += np.dot(dhraw, xs[t].T)
    dWhh += np.dot(dhraw, hs[t-1].T)
    dhnext = np.dot(Whh.T, dhraw)

  for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
    np.clip(dparam, -5, 5, out=dparam)

  return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]


In [ ]:
def sample(h, seed_ix, n):
  """Genera una secuencia de texto desde el modelo."""
  x = np.zeros((vocab_size, 1))
  x[seed_ix] = 1
  ixes = []
  for t in range(n):
    h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)
    y = np.dot(Why, h) + by
    p = np.exp(y) / np.sum(np.exp(y))
    ix = np.random.choice(range(vocab_size), p=p.ravel())
    x = np.zeros((vocab_size, 1))
    x[ix] = 1
    ixes.append(ix)
  return ixes


In [ ]:
n, p = 0, 0
mWxh, mWhh, mWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
mbh, mby = np.zeros_like(bh), np.zeros_like(by)
smooth_loss = -np.log(1.0/vocab_size) * seq_length

while True:
  if p + seq_length + 1 >= len(data) or n == 0:
    hprev = np.zeros((hidden_size,1))
    p = 0

  inputs = [char_to_ix[ch] for ch in data[p:p+seq_length]]
  targets = [char_to_ix[ch] for ch in data[p+1:p+seq_length+1]]

  if n % 100 == 0:
    sample_ix = sample(hprev, inputs[0], 200)
    txt = ''.join(ix_to_char[ix] for ix in sample_ix)
    print('----
%s
----' % (txt, ))

  loss, dWxh, dWhh, dWhy, dbh, dby, hprev = lossFun(inputs, targets, hprev)
  smooth_loss = smooth_loss * 0.999 + loss * 0.001
  if n % 100 == 0: print('iter %d, loss: %f' % (n, smooth_loss))

  for param, dparam, mem in zip([Wxh, Whh, Why, bh, by],
                                [dWxh, dWhh, dWhy, dbh, dby],
                                [mWxh, mWhh, mWhy, mbh, mby]):
    mem += dparam * dparam
    param += -learning_rate * dparam / np.sqrt(mem + 1e-8)

  p += seq_length
  n += 1
